# INF - 404 Inteligencia Artificial

## INF404 IA - Taller RAG local desde cero: embeddings y prompting con LLM

**Docente:** Federico A. Cohen

---

# Chatbot bancario con embeddings y RAG

Notebook autocontenido para una clase introductoria de inteligencia artificial.

Este archivo se puede usar sin README ni archivos externos: los datos, el codigo, las explicaciones y las salidas esperadas estan dentro del notebook.

Vamos a construir un chatbot educativo inspirado en Banco Nacion, usando datos ficticios y simples:

- Caja de ahorro.
- Cuenta corriente.
- Tarjeta de debito.
- Prestamo personal.
- Home Banking y BNA+.
- Sucursales ficticias de Cordoba.

**Importante:** no es informacion bancaria oficial. Es una demo pedagogica para entender RAG.

En esta version no usamos Docker ni Qdrant. Hacemos la busqueda vectorial en memoria para que sea mas facil de aprender. Conceptualmente, Qdrant haria el mismo trabajo de busqueda vectorial, pero como base de datos especializada.

## Que vamos a aprender

- Que es un documento y que es un fragmento.
- Que es un embedding.
- Como una pregunta se transforma en vector.
- Como se recuperan fragmentos parecidos.
- Como se arma un prompt con contexto.
- Como evitar que el chatbot invente respuestas.
- Como medir resultados con metricas simples.

## 0. Prerrequisitos

Para ejecutar este notebook necesitamos:

- Python 3.10 o superior.
- Jupyter Notebook.
- Conexion a internet la primera vez, para instalar paquetes y descargar el modelo de embeddings.
- Un entorno con memoria suficiente para cargar `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, que es un modelo chico para embeddings multilingues.
- Una sesion de Colab gratuita alcanza para la demo. GPU es opcional, pero acelera la generacion con la LLM.

No necesitamos Docker, Qdrant ni base de datos externa para esta clase.

## Nota para Google Colab

Este notebook tambien puede correrse en Google Colab.

Recomendaciones:

- Usar `Entorno de ejecucion > Ejecutar todo` si se quiere correr la clase completa.
- La primera ejecucion puede tardar porque instala librerias y descarga el modelo de embeddings.
- No hace falta montar Google Drive: los documentos estan dentro del notebook.
- No hace falta GPU. CPU alcanza para esta demo chica.
- El modelo de embeddings elegido es pequeno para una sesion gratuita y devuelve vectores de 384 dimensiones.
- Si aparece un aviso de reiniciar el entorno despues de instalar paquetes, reiniciar y volver a ejecutar desde la celda de imports.
- La seccion de LLM real descarga Gemma con `transformers` y genera localmente. No usa APIs externas de chat.

La parte principal de la clase funciona sin API externa porque incluye un chatbot simulado.

## Como usar este notebook

Recomendacion para alumnos:

1. Ejecutar las celdas en orden, desde arriba hacia abajo.
2. Leer las celdas Markdown antes de ejecutar el codigo.
3. Comparar la salida real con la seccion **Salida esperada**.
4. Modificar una pregunta y volver a ejecutar la celda.
5. Al final, cambiar un documento y observar por que hay que recalcular embeddings.

Si una celda de instalacion tarda, es normal: la primera vez se descargan paquetes y el modelo de embeddings.

Si una celda de LLM real tarda, es normal: se descarga el modelo. La parte de embeddings y retrieval ya queda validada antes de llegar a la LLM.

## Mapa de la clase

```text
documentos ficticios del banco
        |
        v
fragmentos de texto
        |
        v
embeddings
        |
        v
busqueda por similitud
        |
        v
contexto recuperado
        |
        v
prompt del chatbot
        |
        v
respuesta o derivacion
        |
        v
evaluacion con metricas
```

## Glosario minimo

**Documento:** texto con informacion. En esta clase lo representamos como JSON.

**Fragmento:** parte chica de un documento. Es la unidad que vamos a recuperar.

**Embedding:** lista de numeros que representa el significado de un texto.

**Busqueda semantica:** busqueda por significado aproximado, no solo por coincidencia exacta de palabras.

**RAG:** tecnica donde primero recuperamos informacion relevante y despues generamos una respuesta usando ese contexto.

**Fuera de alcance:** pregunta que el chatbot no debe responder porque requiere datos personales, informacion actualizada o una operacion real.

## 1. Instalacion de paquetes

El notebook asume que puede preparar su propio entorno.

Ejecutar esta celda al comienzo. Si los paquetes ya estan instalados, no pasa nada.

In [ ]:
%pip install -q "numpy>=1.26,<2.3" sentence-transformers==3.3.1 transformers==4.57.6 accelerate==1.12.0

## 2. Imports

Usamos:

- `sentence-transformers` para convertir texto en embeddings.
- `numpy` para calcular similitud entre vectores.
- `transformers` y `torch` para cargar una LLM local chica tipo Gemma.

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from textwrap import dedent

NOMBRE_MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Modelo de embeddings:", NOMBRE_MODELO_EMBEDDINGS)

**Salida esperada:**

```text
Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

## 3. Documentos JSON dentro del notebook

Estos datos simulan una pequeña base de conocimiento bancaria.

En una aplicacion real, estos textos podrian venir de PDFs, una web publica, documentos internos o una base documental.

Los dejamos en el notebook para que la clase sea autocontenida y los alumnos puedan modificarlos facilmente.

In [ ]:
documentos_banco = [
    {
        "fuente": "cuentas.json",
        "categoria": "productos",
        "titulo": "Productos de cuentas - Banco Nacion educativo",
        "secciones": [
            {
                "titulo_seccion": "Caja de ahorro",
                "texto": "La caja de ahorro es una cuenta bancaria simple para guardar dinero, recibir transferencias, extraer efectivo con tarjeta de debito y operar por home banking. Para abrir una caja de ahorro se solicita DNI vigente, constancia de CUIL o CUIT y datos de contacto actualizados. En esta demo no se informan costos ni tasas reales."
            },
            {
                "titulo_seccion": "Cuenta corriente",
                "texto": "La cuenta corriente esta orientada a personas o comercios que necesitan una operatoria mas frecuente. Puede permitir el uso de cheques y otros servicios asociados, segun evaluacion del banco. Para solicitarla se requiere DNI, constancia de CUIL o CUIT, domicilio declarado, actividad economica y documentacion que justifique ingresos o actividad comercial."
            },
            {
                "titulo_seccion": "Diferencia entre caja de ahorro y cuenta corriente",
                "texto": "La caja de ahorro sirve para operaciones cotidianas simples, como recibir dinero, transferir y extraer efectivo. La cuenta corriente esta pensada para una operatoria mas comercial o intensiva, y puede incluir cheques u otros servicios sujetos a aprobacion."
            }
        ]
    },
    {
        "fuente": "canales_y_saldo.json",
        "categoria": "canales",
        "titulo": "Canales digitales y consultas de saldo",
        "secciones": [
            {
                "titulo_seccion": "Consulta de saldo",
                "texto": "El chatbot educativo no puede consultar saldos, movimientos ni datos personales. Si una persona pregunta cuanto saldo tengo, cual es mi saldo o quiere ver el saldo de su cuenta, debe ingresar a Home Banking, BNA+, un cajero automatico o consultar por los canales oficiales del banco."
            },
            {
                "titulo_seccion": "Home Banking",
                "texto": "Home Banking permite consultar saldos y movimientos, realizar transferencias, pagar servicios y administrar productos bancarios. Para operar se requiere usuario, clave y los mecanismos de seguridad definidos por el banco."
            },
            {
                "titulo_seccion": "BNA+",
                "texto": "BNA+ es un canal digital para operar desde el celular. Permite consultar informacion de cuentas, realizar pagos, transferencias y otras operaciones habilitadas. El chatbot solo puede orientar; no reemplaza el ingreso seguro a la aplicacion."
            }
        ]
    },
    {
        "fuente": "sucursales_cordoba.json",
        "categoria": "sucursales",
        "titulo": "Sucursales de Cordoba - datos ficticios para clase",
        "secciones": [
            {
                "titulo_seccion": "Cordoba Capital - Centro",
                "texto": "Sucursal Cordoba Centro: Av. General Paz 120, Cordoba Capital. Atiende consultas generales, apertura de cuentas, tarjetas y orientacion sobre canales digitales. Horario ficticio para clase: lunes a viernes de 10 a 15."
            },
            {
                "titulo_seccion": "Cordoba Capital - Nueva Cordoba",
                "texto": "Sucursal Nueva Cordoba: Bv. San Juan 540, Cordoba Capital. Atiende productos para personas, cajas de ahorro, cuentas sueldo y consultas de home banking. Horario ficticio para clase: lunes a viernes de 10 a 15."
            },
            {
                "titulo_seccion": "Villa Carlos Paz",
                "texto": "Sucursal Villa Carlos Paz: Av. San Martin 890, Villa Carlos Paz, Cordoba. Atiende consultas de cuentas, tarjetas y prestamos personales. Horario ficticio para clase: lunes a viernes de 10 a 15."
            },
            {
                "titulo_seccion": "Rio Cuarto",
                "texto": "Sucursal Rio Cuarto: Sobremonte 650, Rio Cuarto, Cordoba. Atiende productos para personas y comercios, cuenta corriente, caja de ahorro y canales digitales. Horario ficticio para clase: lunes a viernes de 10 a 15."
            }
        ]
    },
    {
        "fuente": "prestamos_y_tarjetas.json",
        "categoria": "productos",
        "titulo": "Prestamos y tarjetas - Banco Nacion educativo",
        "secciones": [
            {
                "titulo_seccion": "Prestamo personal",
                "texto": "El prestamo personal permite solicitar dinero para distintos fines, sujeto a evaluacion crediticia. Para orientacion general se pueden pedir DNI, ingresos demostrables, antiguedad laboral o actividad declarada y datos de contacto. La aprobacion, el monto, la cuota y la tasa no se definen por chatbot."
            },
            {
                "titulo_seccion": "Tarjeta de debito",
                "texto": "La tarjeta de debito se asocia a una cuenta y permite compras, extracciones y operaciones en cajeros. Si se pierde la tarjeta, se debe bloquear por los canales oficiales y solicitar reposicion por Home Banking, BNA+, telefono oficial o sucursal."
            }
        ]
    },
    {
        "fuente": "fuera_de_alcance.json",
        "categoria": "politicas",
        "titulo": "Politicas de respuesta del chatbot educativo",
        "secciones": [
            {
                "titulo_seccion": "Inflacion y variables economicas",
                "texto": "El chatbot educativo no informa inflacion, cotizacion del dolar, tasas vigentes ni predicciones economicas. Esos datos cambian con el tiempo y deben consultarse en fuentes oficiales o canales actualizados."
            },
            {
                "titulo_seccion": "Datos personales y operaciones",
                "texto": "El chatbot no puede aprobar prestamos, abrir cuentas, consultar saldo, modificar datos personales ni ejecutar operaciones bancarias. Solo puede orientar usando la informacion de los documentos."
            },
            {
                "titulo_seccion": "Respuesta segura",
                "texto": "Cuando la consulta pida datos personales, saldo, movimientos, claves o una operacion bancaria, la respuesta debe derivar a Home Banking, BNA+, cajero automatico, canal oficial o sucursal, segun corresponda."
            }
        ]
    }
]

print("Documentos cargados:", len(documentos_banco))

**Salida esperada:**

```text
Documentos cargados: 5
```

## 4. De documentos a fragmentos

En RAG normalmente no buscamos documentos enteros. Buscamos fragmentos chicos.

Cada seccion se convierte en un fragmento con texto y metadata.

In [ ]:
fragmentos = []

for documento in documentos_banco:
    for numero_seccion, seccion in enumerate(documento["secciones"]):
        fragmento = {
            "id": f"{documento['fuente']}::{numero_seccion}",
            "fuente": documento["fuente"],
            "categoria": documento["categoria"],
            "titulo": documento["titulo"],
            "titulo_seccion": seccion["titulo_seccion"],
            "texto": f"{documento['titulo']} - {seccion['titulo_seccion']}\n{seccion['texto']}",
        }
        fragmentos.append(fragmento)

print("Fragmentos generados:", len(fragmentos))
print("Primer fragmento:")
print(fragmentos[0]["texto"])

**Salida esperada:**

```text
Fragmentos generados: 15
Primer fragmento:
Productos de cuentas - Banco Nacion educativo - Caja de ahorro
La caja de ahorro es una cuenta bancaria simple...
```

## 5. Creamos embeddings

Un embedding es una lista de numeros que representa un texto.

La intuicion:

- Textos parecidos quedan cerca.
- Textos distintos quedan mas lejos.
- Luego podemos buscar por significado, no solo por palabras exactas.

In [ ]:
modelo_embeddings = SentenceTransformer(NOMBRE_MODELO_EMBEDDINGS)

textos_fragmentos = [fragmento["texto"] for fragmento in fragmentos]
matriz_embeddings = modelo_embeddings.encode(textos_fragmentos, normalize_embeddings=True)

print("Cantidad de embeddings:", len(matriz_embeddings))
print("Dimension de cada embedding:", len(matriz_embeddings[0]))
print("Primeros 10 numeros del primer embedding:")
print([round(float(numero), 4) for numero in matriz_embeddings[0][:10]])

**Salida esperada:**

```text
Cantidad de embeddings: 15
Dimension de cada embedding: 384
Primeros 10 numeros del primer embedding:
[... numeros decimales ...]
```

## 6. Busqueda vectorial en memoria

Como los embeddings estan normalizados, podemos usar producto punto como similitud coseno.

Esto reemplaza a Qdrant para la clase:

```text
pregunta -> embedding -> comparar contra todos los fragmentos -> top K
```

In [ ]:
def buscar_fragmentos(pregunta_usuario, cantidad_resultados=4):
    vector_pregunta = modelo_embeddings.encode(pregunta_usuario, normalize_embeddings=True)
    puntajes = matriz_embeddings @ vector_pregunta
    indices_ordenados = np.argsort(puntajes)[::-1][:cantidad_resultados]
    
    resultados = []
    for posicion, indice in enumerate(indices_ordenados, start=1):
        resultado = {
            "posicion": posicion,
            "puntaje": float(puntajes[indice]),
            "fragmento": fragmentos[indice],
        }
        resultados.append(resultado)
    
    return vector_pregunta, resultados

## 7. El paso intermedio de la query

Esta celda muestra lo que normalmente queda escondido en una app:

1. Pregunta original.
2. Vector de la pregunta.
3. Parametros de busqueda.
4. Fragmentos recuperados.

Este es uno de los momentos mas importantes para enseñar RAG.

In [ ]:
def mostrar_paso_intermedio(pregunta_usuario, cantidad_resultados=4):
    vector_pregunta, resultados = buscar_fragmentos(pregunta_usuario, cantidad_resultados)

    print("PREGUNTA DEL USUARIO")
    print(pregunta_usuario)
    print("\nEMBEDDING DE LA PREGUNTA")
    print("Dimension:", len(vector_pregunta))
    print("Primeros 10 numeros:", [round(float(numero), 4) for numero in vector_pregunta[:10]])
    print("\nPARAMETROS DE BUSQUEDA")
    print("cantidad_resultados:", cantidad_resultados)
    print("metrica: similitud coseno")
    print("base vectorial: memoria local con numpy")
    print("\nFRAGMENTOS RECUPERADOS")
    
    for resultado in resultados:
        fragmento = resultado["fragmento"]
        print("-" * 90)
        print(f"[{resultado['posicion']}] puntaje={resultado['puntaje']:.3f}")
        print("fuente:", fragmento["fuente"])
        print("seccion:", fragmento["titulo_seccion"])
        print(fragmento["texto"])
    
    return resultados

resultados_cuenta_corriente = mostrar_paso_intermedio(
    "Que requisitos necesito para abrir una cuenta corriente?",
    cantidad_resultados=4,
)

**Salida esperada:**

```text
PREGUNTA DEL USUARIO
Que requisitos necesito para abrir una cuenta corriente?

EMBEDDING DE LA PREGUNTA
Dimension: 384
Primeros 10 numeros: [...]

PARAMETROS DE BUSQUEDA
cantidad_resultados: 4
metrica: similitud coseno
base vectorial: memoria local con numpy

FRAGMENTOS RECUPERADOS
[1] puntaje=...
fuente: cuentas.json
seccion: Cuenta corriente
...
```

## 8. Probamos varias preguntas

Estas preguntas deberian recuperar fragmentos correctos.

In [ ]:
preguntas_de_prueba = [
    "Que necesito para abrir una caja de ahorro?",
    "Cual es la diferencia entre caja de ahorro y cuenta corriente?",
    "Que sucursales hay en Cordoba?",
    "Perdi mi tarjeta de debito, que hago?",
]

for pregunta in preguntas_de_prueba:
    print("\n" + "=" * 100)
    mostrar_paso_intermedio(pregunta, cantidad_resultados=2)

**Salida esperada:**

- La pregunta sobre caja de ahorro deberia traer la seccion `Caja de ahorro`.
- La pregunta sobre diferencias deberia traer `Diferencia entre caja de ahorro y cuenta corriente`.
- La pregunta sobre Cordoba deberia traer sucursales.
- La pregunta sobre tarjeta perdida deberia traer `Tarjeta de debito`.

## 9. Prompt del chatbot

El prompt define el comportamiento del chatbot.

La idea principal es: **responder solo con el contexto recuperado y no inventar**.

In [ ]:
PROMPT_SISTEMA = """
Sos un chatbot educativo de un banco ficticio inspirado en Banco Nacion.
Tu objetivo es ayudar a estudiantes a entender RAG con productos bancarios simples.

Reglas obligatorias:
- Responde solo con informacion que aparezca en el contexto recuperado.
- No inventes tasas, costos, requisitos, horarios, direcciones ni condiciones.
- No consultes ni simules consultar datos personales, saldos, movimientos o aprobaciones.
- Si preguntan por saldo, movimientos, claves u operaciones, deriva a Home Banking, BNA+, cajero, canal oficial o sucursal.
- Si preguntan por inflacion, dolar, tasas vigentes o predicciones, explica que son datos actualizados fuera de esta base documental.
- Si el contexto no alcanza, deci claramente: 'No tengo informacion suficiente en los documentos cargados'.
- Responde en espanol claro, con tono amable y didactico.
- Cita las fuentes usando numeros de fragmento, por ejemplo [1] o [2].
- Al final agrega: 'Nota: demo educativa, no informacion bancaria oficial'.
""".strip()

def construir_contexto(resultados):
    bloques = []
    for resultado in resultados:
        fragmento = resultado["fragmento"]
        bloques.append(
            f"[{resultado['posicion']}] fuente={fragmento['fuente']} | seccion={fragmento['titulo_seccion']}\n"
            f"{fragmento['texto']}"
        )
    return "\n\n".join(bloques)

def construir_prompt_usuario(pregunta_usuario, resultados):
    contexto = construir_contexto(resultados)
    return dedent(f"""
    Pregunta del usuario:
    {pregunta_usuario}

    Contexto recuperado por RAG:
    {contexto}

    Instrucciones de respuesta:
    - Responde breve y claro.
    - Usa solo el contexto recuperado.
    - Cita la fuente con [1], [2], etc.
    - Si la pregunta esta fuera de alcance, explica por que y deriva al canal correspondiente.
    """).strip()

prompt_usuario = construir_prompt_usuario(
    "Que requisitos necesito para abrir una cuenta corriente?",
    resultados_cuenta_corriente,
)

print("PROMPT DE SISTEMA:\n")
print(PROMPT_SISTEMA)
print("\n" + "=" * 100 + "\n")
print("PROMPT DE USUARIO:\n")
print(prompt_usuario)

**Salida esperada:**

Se ve el prompt completo que recibira una LLM:

- Reglas del chatbot.
- Pregunta del usuario.
- Fragmentos recuperados.
- Instrucciones para no inventar.

## 10. Primero: un chatbot ingenuo que alucina

Antes de usar guardrails, vamos a simular un chatbot mal diseñado.

Este chatbot toma el primer fragmento recuperado y responde aunque el contexto no alcance.

Esto es peligroso porque puede:

- Contestar una pregunta fuera de alcance.
- Citar una fuente que no sostiene la respuesta.
- Dar una sensacion falsa de seguridad.

Lo mostramos a proposito para reconocer el problema.


In [ ]:
PROMPT_INGENUO = """
Sos un asistente bancario amable. Responde siempre de forma util al usuario.
Si falta informacion, usa tu mejor criterio para completar la respuesta.
""".strip()


def contiene_alguna_palabra_ingenua(texto, palabras):
    texto_normalizado = texto.lower()
    return any(palabra in texto_normalizado for palabra in palabras)

def responder_de_forma_ingenua(pregunta_usuario, resultados):
    mejor_fragmento = resultados[0]["fragmento"]
    
    if contiene_alguna_palabra_ingenua(pregunta_usuario, ["inflacion", "dolar", "tasa"]):
        return (
            "La inflacion de este mes podria ubicarse alrededor de un valor estimado, "
            "pero te recomiendo verificarlo en fuentes oficiales. "
            f"Segun el contexto recuperado: {mejor_fragmento['texto']} [1]"
        )
    
    if contiene_alguna_palabra_ingenua(pregunta_usuario, ["saldo", "cuanto tengo"]):
        return (
            "Tu saldo puede consultarse desde los canales digitales del banco. "
            f"Segun el contexto recuperado: {mejor_fragmento['texto']} [1]"
        )
    
    return (
        "Con la informacion disponible, la respuesta seria: "
        f"{mejor_fragmento['texto']} [1]"
    )

def chatbot_ingenuo(pregunta_usuario, cantidad_resultados=3):
    resultados = mostrar_paso_intermedio(pregunta_usuario, cantidad_resultados)
    print("\n" + "=" * 100)
    print("RESPUESTA DEL CHATBOT INGENUO")
    print(responder_de_forma_ingenua(pregunta_usuario, resultados))

chatbot_ingenuo("Cual es la inflacion de este mes?", cantidad_resultados=3)


**Que deberia observarse:**

El chatbot ingenuo intenta responder una pregunta que requiere informacion actualizada.

Aunque diga "verificar en fuentes oficiales", ya insinuo una respuesta sin evidencia. Eso es una alucinacion.

El problema no es solo el modelo: tambien es el diseño del prompt y de las reglas de respuesta.


## 11. Agregamos guardrails

Ahora reformulamos el comportamiento esperado.

Un guardrail es una regla que limita lo que el chatbot puede hacer.

En este caso, el guardrail central es:

> Si la respuesta no esta en el contexto recuperado, no responder. Derivar o decir que falta evidencia.

Esto no hace perfecto al sistema, pero reduce respuestas inventadas.


## 12. Chatbot simulado con guardrails

Para que la clase funcione incluso antes de descargar Gemma, simulamos la respuesta del chatbot.

Esto no reemplaza a una LLM real, pero sirve para enseñar el comportamiento esperado:

- Responder cuando hay evidencia.
- Derivar cuando se pregunta por saldo.
- Rechazar cuando se pregunta por inflacion, dolar o tasas vigentes.
- No aprobar operaciones.

In [ ]:
def contiene_alguna_palabra(texto, palabras):
    texto_normalizado = texto.lower()
    return any(palabra in texto_normalizado for palabra in palabras)

def posicion_fragmento_relevante(resultados, palabras_clave):
    for resultado in resultados:
        fragmento = resultado["fragmento"]
        texto_fragmento = f"{fragmento['titulo_seccion']} {fragmento['texto']}".lower()
        if contiene_alguna_palabra(texto_fragmento, palabras_clave):
            return resultado["posicion"]
    return None

def respuesta_sin_evidencia_recuperada(tipo_consulta):
    return (
        f"Detecto que la consulta parece ser sobre {tipo_consulta}, pero el contexto recuperado no contiene "
        "un fragmento suficientemente especifico para responder con fuente. En un sistema RAG honesto, no debo citar "
        "ni completar la respuesta si la evidencia no esta en el contexto. Conviene aumentar el top_k, mejorar los documentos "
        "o ajustar la estrategia de recuperacion.\n\n"
        "Nota: demo educativa, no informacion bancaria oficial."
    )

def responder_de_forma_simulada(pregunta_usuario, resultados):
    pregunta = pregunta_usuario.lower()
    mejor_fragmento = resultados[0]["fragmento"]
    
    if contiene_alguna_palabra(pregunta, ["saldo", "movimientos", "cuanto tengo"]):
        posicion_fuente = posicion_fragmento_relevante(resultados, ["saldo", "saldos", "bna+", "cajero"])
        if posicion_fuente is None:
            return respuesta_sin_evidencia_recuperada("consulta de saldo")
        return (
            "No puedo consultar saldos, movimientos ni datos personales desde este chatbot. "
            f"Para ver el saldo, ingresa a Home Banking, BNA+, un cajero automatico o un canal oficial del banco [{posicion_fuente}].\n\n"
            "Nota: demo educativa, no informacion bancaria oficial."
        )
    
    if contiene_alguna_palabra(pregunta, ["inflacion", "dolar", "tasa vigente", "tasas vigentes", "prediccion"]):
        posicion_fuente = posicion_fragmento_relevante(resultados, ["inflacion", "dolar", "tasas vigentes", "predicciones"])
        if posicion_fuente is None:
            return respuesta_sin_evidencia_recuperada("dato economico actualizado")
        return (
            "No tengo informacion suficiente en los documentos cargados para responder ese dato actualizado. "
            f"Inflacion, dolar, tasas vigentes y predicciones economicas deben consultarse en fuentes oficiales o canales actualizados [{posicion_fuente}].\n\n"
            "Nota: demo educativa, no informacion bancaria oficial."
        )
    
    if contiene_alguna_palabra(pregunta, ["aprobas", "aprobar", "aprobame"]):
        posicion_fuente = posicion_fragmento_relevante(resultados, ["prestamo", "aprobar", "aprobacion", "operaciones bancarias"])
        if posicion_fuente is None:
            return respuesta_sin_evidencia_recuperada("operacion bancaria o aprobacion")
        return (
            "No puedo aprobar prestamos ni ejecutar operaciones bancarias. "
            f"Solo puedo orientar con informacion general de los documentos. Para avanzar, consulta un canal oficial o una sucursal [{posicion_fuente}].\n\n"
            "Nota: demo educativa, no informacion bancaria oficial."
        )
    
    return (
        "Segun la informacion recuperada, la respuesta esta en la seccion "
        f"'{mejor_fragmento['titulo_seccion']}'.\n\n"
        f"{mejor_fragmento['texto']} [1]\n\n"
        "Nota: demo educativa, no informacion bancaria oficial."
    )

def chatbot_simulado(pregunta_usuario, cantidad_resultados=4, mostrar_prompt=False):
    resultados = mostrar_paso_intermedio(pregunta_usuario, cantidad_resultados)
    prompt_usuario = construir_prompt_usuario(pregunta_usuario, resultados)
    
    if mostrar_prompt:
        print("\n" + "=" * 100)
        print("PROMPT QUE RECIBIRIA LA LLM")
        print(prompt_usuario)
    
    print("\n" + "=" * 100)
    print("RESPUESTA DEL CHATBOT SIMULADO")
    print(responder_de_forma_simulada(pregunta_usuario, resultados))


## 13. Pregunta que debe responder bien

Ejemplo: requisitos para cuenta corriente.

In [ ]:
chatbot_simulado(
    "Que requisitos necesito para abrir una cuenta corriente?",
    cantidad_resultados=4,
    mostrar_prompt=True,
)

**Salida esperada:**

El chatbot recupera la seccion `Cuenta corriente` y responde con requisitos como DNI, CUIL o CUIT, domicilio declarado, actividad economica y documentacion que justifique ingresos o actividad comercial.

## 14. Pregunta de saldo

Esta pregunta no debe responder con un numero inventado. Debe derivar a canales seguros.

In [ ]:
chatbot_simulado("Cuanto saldo tengo en mi cuenta?", cantidad_resultados=4)

**Salida esperada:**

El chatbot debe decir que no puede consultar saldo y derivar a Home Banking, BNA+, cajero automatico o canales oficiales.

## 15. Pregunta fuera del RAG: inflacion

Esta pregunta requiere datos actualizados. El chatbot no debe inventar.

In [ ]:
chatbot_simulado("Cual es la inflacion de este mes?", cantidad_resultados=4)

**Salida esperada:**

El chatbot debe explicar que inflacion es informacion actualizada fuera de la base documental y derivar a fuentes oficiales o canales actualizados.

## 16. LLM real local con Gemma

Hasta aca usamos un chatbot simulado para que la clase sea estable.

Ahora conectamos una LLM real local usando `transformers`. Esto descarga los pesos de Gemma en la sesion y genera texto localmente: no llama a OpenAI, Gemini ni ninguna API externa de chat.

Modelo usado:

- `google/gemma-3-270m-it`: modelo Gemma chico, adecuado para una demo educativa en Colab.

Si Hugging Face solicita aceptar terminos o iniciar sesion para descargar Gemma, hacerlo desde la pagina del modelo o con un token de Hugging Face. Eso es autenticacion para descargar pesos, no una API de generacion.

La idea pedagogica no es lograr la mejor respuesta del mundo, sino mostrar el circuito completo:

```text
pregunta -> retrieval -> contexto -> prompt con guardrails -> Gemma local -> respuesta
```

En Colab, si hay GPU activada, la generacion va mas rapido. Si no hay GPU, tambien puede correr, pero mas lento.


In [ ]:
MODELO_GEMMA = "google/gemma-3-270m-it"

DISPOSITIVO_LLM = "cuda" if torch.cuda.is_available() else "cpu"
TIPO_DATO_LLM = torch.float16 if DISPOSITIVO_LLM == "cuda" else torch.float32

print("Dispositivo para LLM:", DISPOSITIVO_LLM)
print("Modo: descarga de pesos de Gemma con transformers, sin API externa de chat")
print("Modelo Gemma:", MODELO_GEMMA)

tokenizer_llm = AutoTokenizer.from_pretrained(MODELO_GEMMA)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    MODELO_GEMMA,
    torch_dtype=TIPO_DATO_LLM,
    low_cpu_mem_usage=True,
)
modelo_llm.to(DISPOSITIVO_LLM)
modelo_llm.eval()

MODELO_LLM_ACTIVO = MODELO_GEMMA
print("Modelo LLM activo:", MODELO_LLM_ACTIVO)


### Funcion para generar con la LLM

Esta funcion recibe el prompt completo y genera una respuesta.

Usamos `do_sample=False` para que la salida sea mas estable en clase.


In [ ]:
def generar_con_llm_local(prompt_completo, max_new_tokens=180):
    mensajes = [
        {
            "role": "user",
            "content": prompt_completo,
        }
    ]
    
    if getattr(tokenizer_llm, "chat_template", None):
        entrada = tokenizer_llm.apply_chat_template(
            mensajes,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        entrada = tokenizer_llm(prompt_completo, return_tensors="pt")
    
    entrada = {clave: valor.to(DISPOSITIVO_LLM) for clave, valor in entrada.items()}
    longitud_entrada = entrada["input_ids"].shape[-1]
    
    with torch.no_grad():
        salida = modelo_llm.generate(
            **entrada,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_llm.eos_token_id,
        )
    
    tokens_generados = salida[0][longitud_entrada:]
    return tokenizer_llm.decode(tokens_generados, skip_special_tokens=True).strip()


### Chatbot RAG con LLM real

Esta version usa el mismo retrieval y el mismo prompt con guardrails, pero la respuesta final la genera la LLM.


In [ ]:
def chatbot_con_llm_real(pregunta_usuario, cantidad_resultados=4, mostrar_prompt=False):
    resultados = mostrar_paso_intermedio(pregunta_usuario, cantidad_resultados)
    prompt_usuario = construir_prompt_usuario(pregunta_usuario, resultados)
    prompt_completo = PROMPT_SISTEMA + "\n\n" + prompt_usuario
    
    if mostrar_prompt:
        print("\n" + "=" * 100)
        print("PROMPT COMPLETO ENVIADO A LA LLM")
        print(prompt_completo)
    
    print("\n" + "=" * 100)
    print("RESPUESTA DE LA LLM REAL")
    print("Modelo:", MODELO_LLM_ACTIVO)
    print(generar_con_llm_local(prompt_completo))

chatbot_con_llm_real(
    "Cual es la inflacion de este mes?",
    cantidad_resultados=4,
    mostrar_prompt=False,
)


**Salida esperada:**

La LLM deberia rechazar o derivar la pregunta sobre inflacion porque el prompt y el contexto indican que es un dato actualizado fuera de la base documental.

Como es una LLM chica, puede no obedecer perfecto. Eso tambien es material de clase: los guardrails reducen alucinaciones, pero hay que evaluar.


## 17. Comparacion antes/despues: ingenuo vs guardrails

Ahora medimos algo muy simple: ante preguntas fuera de alcance, ¿el chatbot responde o rechaza?

Queremos que el chatbot seguro rechace o derive, no que complete con informacion inventada.


In [ ]:
casos_guardrails = [
    {
        "pregunta": "Cual es la inflacion de este mes?",
        "decision_correcta": "rechazar_o_derivar",
    },
    {
        "pregunta": "A cuanto esta el dolar hoy?",
        "decision_correcta": "rechazar_o_derivar",
    },
    {
        "pregunta": "Cuanto saldo tengo en mi cuenta?",
        "decision_correcta": "rechazar_o_derivar",
    },
    {
        "pregunta": "Me aprobas un prestamo personal?",
        "decision_correcta": "rechazar_o_derivar",
    },
]

def clasificar_decision(texto_respuesta):
    texto = texto_respuesta.lower()
    marcas_rechazo = [
        "no puedo",
        "no tengo informacion suficiente",
        "no debo citar",
        "fuera de esta base documental",
        "debe consultarse",
        "deriva",
        "canal oficial",
    ]
    if any(marca in texto for marca in marcas_rechazo):
        return "rechazar_o_derivar"
    return "responder"

def evaluar_guardrails(casos):
    aciertos_ingenuo = 0
    aciertos_seguro = 0
    filas = []
    
    for caso in casos:
        _, resultados = buscar_fragmentos(caso["pregunta"], cantidad_resultados=4)
        respuesta_ingenua = responder_de_forma_ingenua(caso["pregunta"], resultados)
        respuesta_segura = responder_de_forma_simulada(caso["pregunta"], resultados)
        decision_ingenua = clasificar_decision(respuesta_ingenua)
        decision_segura = clasificar_decision(respuesta_segura)
        
        acierto_ingenuo = decision_ingenua == caso["decision_correcta"]
        acierto_seguro = decision_segura == caso["decision_correcta"]
        aciertos_ingenuo += int(acierto_ingenuo)
        aciertos_seguro += int(acierto_seguro)
        
        filas.append({
            "pregunta": caso["pregunta"],
            "decision_ingenua": decision_ingenua,
            "decision_segura": decision_segura,
            "ingenuo_ok": acierto_ingenuo,
            "seguro_ok": acierto_seguro,
        })
    
    total = len(casos)
    return {
        "accuracy_ingenuo": aciertos_ingenuo / total,
        "accuracy_con_guardrails": aciertos_seguro / total,
    }, filas

metricas_guardrails, filas_guardrails = evaluar_guardrails(casos_guardrails)

print("METRICAS GUARDRAILS")
for nombre, valor in metricas_guardrails.items():
    print(f"{nombre}: {valor:.2%}")

print("\nDETALLE")
for fila in filas_guardrails:
    print("-" * 100)
    print("pregunta:", fila["pregunta"])
    print("ingenuo:", fila["decision_ingenua"], "| ok:", fila["ingenuo_ok"])
    print("seguro:", fila["decision_segura"], "| ok:", fila["seguro_ok"])


**Salida esperada:**

El chatbot ingenuo deberia tener peor accuracy porque intenta responder preguntas fuera de alcance.

El chatbot con guardrails deberia mejorar porque rechaza o deriva cuando no debe responder.


## 18. Evaluacion simple: accuracy y recall

Hasta ahora vimos ejemplos sueltos. Pero en sistemas de IA necesitamos medir.

Vamos a evaluar dos cosas:

1. **Retrieval:** si la busqueda trae el fragmento correcto.
2. **Politica del chatbot:** si responde, deriva o rechaza cuando corresponde.

Metricas simples:

- `precision_top_1`: el primer resultado recuperado era correcto.
- `recuperacion_contexto`: alguno de los fragmentos enviados al chatbot era correcto.
- `accuracy_politica`: el chatbot tomo la decision esperada sin citar evidencia inexistente.

In [ ]:
casos_evaluacion = [
    {
        "pregunta": "Que requisitos necesito para abrir una cuenta corriente?",
        "secciones_esperadas": ["Cuenta corriente"],
        "comportamiento_esperado": "responder_con_contexto",
    },
    {
        "pregunta": "Que necesito para abrir una caja de ahorro?",
        "secciones_esperadas": ["Caja de ahorro"],
        "comportamiento_esperado": "responder_con_contexto",
    },
    {
        "pregunta": "Cual es la diferencia entre caja de ahorro y cuenta corriente?",
        "secciones_esperadas": ["Diferencia entre caja de ahorro y cuenta corriente"],
        "comportamiento_esperado": "responder_con_contexto",
    },
    {
        "pregunta": "Que sucursales hay en Cordoba?",
        "secciones_esperadas": ["Cordoba Capital - Centro", "Cordoba Capital - Nueva Cordoba", "Villa Carlos Paz", "Rio Cuarto"],
        "comportamiento_esperado": "responder_con_contexto",
    },
    {
        "pregunta": "Perdi mi tarjeta de debito, que hago?",
        "secciones_esperadas": ["Tarjeta de debito"],
        "comportamiento_esperado": "responder_con_contexto",
    },
    {
        "pregunta": "Cuanto saldo tengo en mi cuenta?",
        "secciones_esperadas": ["Consulta de saldo", "Respuesta segura"],
        "comportamiento_esperado": "derivar_saldo",
    },
    {
        "pregunta": "Cual es la inflacion de este mes?",
        "secciones_esperadas": ["Inflacion y variables economicas"],
        "comportamiento_esperado": "rechazar_dato_actualizado",
    },
    {
        "pregunta": "Me aprobas un prestamo personal?",
        "secciones_esperadas": ["Prestamo personal", "Datos personales y operaciones"],
        "comportamiento_esperado": "rechazar_operacion",
    },
]

print("Casos de evaluacion:", len(casos_evaluacion))

**Salida esperada:**

```text
Casos de evaluacion: 8
```

In [ ]:
def clasificar_respuesta_simulada(respuesta):
    respuesta_normalizada = respuesta.lower()
    
    if "no puedo consultar saldos" in respuesta_normalizada:
        return "derivar_saldo"
    
    if "inflacion" in respuesta_normalizada and "dato actualizado" in respuesta_normalizada:
        return "rechazar_dato_actualizado"
    
    if "no puedo aprobar prestamos" in respuesta_normalizada:
        return "rechazar_operacion"
    
    return "responder_con_contexto"

def evaluar_sistema(casos):
    aciertos_top_1 = 0
    aciertos_contexto = 0
    aciertos_politica = 0
    filas_resultado = []
    
    for caso in casos:
        _, resultados = buscar_fragmentos(caso["pregunta"], cantidad_resultados=4)
        secciones_recuperadas = [resultado["fragmento"]["titulo_seccion"] for resultado in resultados]
        secciones_esperadas = caso["secciones_esperadas"]
        
        top_1_correcto = secciones_recuperadas[0] in secciones_esperadas
        contexto_correcto = any(seccion in secciones_esperadas for seccion in secciones_recuperadas)
        
        respuesta = responder_de_forma_simulada(caso["pregunta"], resultados)
        comportamiento_obtenido = clasificar_respuesta_simulada(respuesta)
        politica_correcta = comportamiento_obtenido == caso["comportamiento_esperado"]
        
        aciertos_top_1 += int(top_1_correcto)
        aciertos_contexto += int(contexto_correcto)
        aciertos_politica += int(politica_correcta)
        
        filas_resultado.append({
            "pregunta": caso["pregunta"],
            "top_1": secciones_recuperadas[0],
            "top_1_ok": top_1_correcto,
            "contexto_ok": contexto_correcto,
            "politica_esperada": caso["comportamiento_esperado"],
            "politica_obtenida": comportamiento_obtenido,
            "politica_ok": politica_correcta,
        })
    
    total = len(casos)
    metricas = {
        "precision_top_1": aciertos_top_1 / total,
        "recuperacion_contexto": aciertos_contexto / total,
        "accuracy_politica": aciertos_politica / total,
    }
    
    return metricas, filas_resultado

metricas, filas_resultado = evaluar_sistema(casos_evaluacion)

print("METRICAS")
for nombre_metrica, valor in metricas.items():
    print(f"{nombre_metrica}: {valor:.2%}")

print("\nDETALLE POR CASO")
for fila in filas_resultado:
    print("-" * 100)
    print("pregunta:", fila["pregunta"])
    print("top_1:", fila["top_1"], "| ok:", fila["top_1_ok"])
    print("contexto_ok:", fila["contexto_ok"])
    print("politica:", fila["politica_obtenida"], "| ok:", fila["politica_ok"])

**Salida esperada:**

Los porcentajes pueden variar un poco segun version de librerias/modelo, pero deberiamos ver algo cercano a:

```text
METRICAS
precision_top_1: ...%
recuperacion_contexto: ...%
accuracy_politica: ...%
```

Como explicarlo:

- Si `precision_top_1` es baja, el primer fragmento no siempre es el mejor.
- Si `recuperacion_contexto` es alta, el contexto correcto aparece entre los fragmentos que recibe el chatbot.
- Si `accuracy_politica` es alta, las reglas de derivacion/rechazo estan funcionando sin inventar fuentes.

Nota pedagogica: es normal que alguna pregunta no tenga retrieval perfecto. Por ejemplo, una pregunta de saldo puede parecerse semanticamente a textos de cuentas. Eso sirve para explicar que en sistemas reales se ajustan documentos, prompts, filtros, top K y evaluaciones.

## 19. Mini desafio

Modificar `documentos_banco` en la celda 3:

- Agregar una sucursal ficticia.
- Agregar un requisito nuevo para cuenta corriente.
- Agregar una politica nueva de fuera de alcance.

Despues volver a ejecutar desde la celda 4.

Pregunta para pensar:

> Por que hay que recalcular embeddings cuando cambia un documento?